# 15 Pi0 端到端：权限、长训、严格输入与失败桶

            Pi0 这一版不是“完全复现历史 hard8”的主线，而是进阶诊断模型。Notebook 重点展示 gated 权限、长训配置、S8500 checkpoint、median reducer、严格评估和失败桶分析。


In [1]:

from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

try:
    from IPython.display import HTML, Image, Markdown, Video, display
except Exception:
    class Markdown(str):
        pass

    class HTML(str):
        pass

    def Image(filename=None, width=None, **kwargs):
        return f"[image] {filename}"

    def Video(filename=None, embed=False, width=None, **kwargs):
        return f"[video] {filename}"

    def display(obj):
        print(obj)


def find_topic_root():
    override = (
        os.environ.get("AMD_TOPIC_ROOT")
        or os.environ.get("NOTEBOOK_TOPIC_ROOT")
        or os.environ.get("TOPIC_ROOT")
    )
    roots = [Path(override).expanduser()] if override else []
    cwd = Path.cwd().resolve()
    roots.extend([cwd, *cwd.parents])

    candidates = []
    for root in roots:
        candidates.extend(
            [
                root,
                root / "16-专题组队学习" / "04-AMD-ROCm策略复刻专题",
                root / "04-AMD-ROCm策略复刻专题",
            ]
        )
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError(
        "找不到 AMD ROCm 专题目录。请从仓库根目录、专题目录启动 Jupyter，"
        "或设置 AMD_TOPIC_ROOT。"
    )


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", TOPIC_ROOT / "external" / "04mujoco复现ACT、Pi0、SmolVLA")
).expanduser()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", TOPIC_ROOT / "data")).expanduser()
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", TOPIC_ROOT / "checkpoints")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))

# The AMD teaching workflow should be runnable from local datasets/checkpoints.
# Avoid surprising network calls during class or when AUP/Radeon Cloud cannot
# reach Hugging Face.
os.environ.setdefault("HF_HUB_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("TRANSFORMERS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_DATASETS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_HOME", str(Path(os.environ.get("CACHE_ROOT", OUTPUT_ROOT / "cache")) / "huggingface"))
os.environ.setdefault("HF_DATASETS_CACHE", str(Path(os.environ["HF_HOME"]) / "datasets"))

def public_path(path):
    path = Path(path)
    replacements = [
        (TOPIC_ROOT, "$TOPIC_ROOT"),
        (PROJECT_ROOT, "$PROJECT_ROOT"),
        (DATA_ROOT, "$DATA_ROOT"),
        (MODEL_ROOT, "$MODEL_ROOT"),
        (OUTPUT_ROOT, "$OUTPUT_ROOT"),
    ]
    value = str(path)
    for root, label in sorted(replacements, key=lambda item: len(str(item[0])), reverse=True):
        root_value = str(root)
        if root_value and value.startswith(root_value):
            return label + value[len(root_value):]
    return value


print("TOPIC_ROOT =", public_path(TOPIC_ROOT))
print("PROJECT_ROOT =", public_path(PROJECT_ROOT))
print("DATA_ROOT =", public_path(DATA_ROOT))
print("MODEL_ROOT =", public_path(MODEL_ROOT))
print("OUTPUT_ROOT =", public_path(OUTPUT_ROOT))


TOPIC_ROOT = $TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $DATA_ROOT
MODEL_ROOT = $MODEL_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT


In [2]:

def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(public_path(x) if isinstance(x, (str, Path)) else str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


def show_json(path, max_chars=5000):
    path = Path(path)
    if not path.exists():
        print("文件不存在：", public_path(path))
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))
    return data


def show_video(filename, title):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        try:
            display(Video(filename=str(path), embed=True, width=960, html_attributes="controls muted"))
        except TypeError:
            display(Video(filename=str(path), embed=True, width=960))
    else:
        print("缺少视频素材：", public_path(path))


def show_image(filename, title, width=960):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        try:
            display(Image(filename=str(path), width=width))
        except TypeError:
            display(Image(filename=str(path)))
    else:
        print("缺少图片素材：", public_path(path))


def run_cmd_preview(command, cwd=None):
    shown = [public_path(x) if isinstance(x, (str, Path)) else x for x in command]
    print("$", shlex.join([str(x) for x in shown]))
    if cwd:
        print("cwd =", public_path(cwd))


def tail_log(log_path, lines=40):
    path = Path(log_path)
    if not path.exists():
        print("日志不存在：", public_path(path))
        return
    content = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(content[-lines:]))


def env_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_SMOKE = env_flag("RUN_SMOKE")
RUN_LONG_TRAIN = env_flag("RUN_LONG_TRAIN")
RUN_EVAL = env_flag("RUN_EVAL")
EVAL_SCRIPT = Path(os.environ.get("EVAL_SCRIPT", PROJECT_ROOT / "eval_policy_success.py"))


_XVFB_PROCESS = None


def ensure_xvfb_display():
    """Start a lightweight virtual display for headless MuJoCo evaluation."""
    global _XVFB_PROCESS
    if os.environ.get("DISPLAY"):
        print("DISPLAY =", os.environ["DISPLAY"])
        return None
    xvfb_bin = shutil.which("Xvfb")
    if not xvfb_bin:
        print("没有发现 Xvfb；如遇 GLFW DISPLAY 报错，请先安装 xvfb。")
        return None
    display_id = os.environ.get("NOTEBOOK_XVFB_DISPLAY", ":99")
    _XVFB_PROCESS = subprocess.Popen(
        [xvfb_bin, display_id, "-screen", "0", "1280x1024x24"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    os.environ["DISPLAY"] = display_id
    print("已启动 Notebook 内部 Xvfb：DISPLAY =", display_id)
    return _XVFB_PROCESS


def ensure_project_layout():
    required = [PROJECT_ROOT / "asset" / "example_scene_y2.xml", PROJECT_ROOT / "mujoco_env"]
    missing = [path for path in required if not path.exists()]
    if missing:
        print("当前 PROJECT_ROOT 还不是可运行工程，缺少：")
        for path in missing:
            print(" -", public_path(path))
        print("请先设置 PROJECT_ROOT，再运行训练或评估单元。")
        return False
    return True


def write_json_yaml(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import yaml
        text = yaml.safe_dump(payload, allow_unicode=True, sort_keys=False)
    except Exception:
        text = json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
    path.write_text(text, encoding="utf-8")
    print("写出配置：", public_path(path))
    return path


def make_lerobot_train_config(policy_type, dataset_repo_id, dataset_root, output_dir, steps, batch_size, chunk_size, n_action_steps, seed=42):
    save_freq = int(os.environ.get(f"{policy_type.upper()}_SAVE_FREQ", os.environ.get("SAVE_FREQ", str(steps))))
    return {
        "dataset": {
            "repo_id": dataset_repo_id,
            "root": str(dataset_root),
            "use_imagenet_stats": True,
        },
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "output_dir": str(output_dir),
        "job_name": Path(output_dir).name,
        "batch_size": int(batch_size),
        "steps": int(steps),
        "save_freq": max(1, save_freq),
        "log_freq": 20,
        "num_workers": 4,
        "seed": int(seed),
        "resume": False,
        "eval_freq": -1,
        "save_checkpoint": True,
        "use_policy_training_preset": True,
        "wandb": {"enable": False, "disable_artifact": True},
    }


def train_lerobot_config_in_notebook(config_path, enabled=False, progress_name="train"):
    """Run LeRobot offline training directly inside the notebook kernel.

    The notebook cell owns dataset creation, policy creation, optimizer steps,
    checkpoint saving, tqdm progress, and metric JSONL writing.
    """
    config_path = Path(config_path)
    print("config =", public_path(config_path))
    if not enabled:
        print("未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。")
        return None
    if not ensure_project_layout():
        return None

    import time
    from contextlib import nullcontext

    import draccus
    import torch
    from torch.amp import GradScaler
    from tqdm.auto import tqdm

    from lerobot.common.datasets.factory import make_dataset
    from lerobot.common.datasets.sampler import EpisodeAwareSampler
    from lerobot.common.optim.factory import make_optimizer_and_scheduler
    from lerobot.common.policies.factory import make_policy
    from lerobot.common.policies.utils import get_device_from_parameters
    from lerobot.common.utils.random_utils import set_seed
    from lerobot.common.utils.train_utils import get_step_checkpoint_dir, save_checkpoint, update_last_checkpoint
    from lerobot.common.utils.utils import get_safe_torch_device
    from lerobot.configs.train import TrainPipelineConfig

    cfg = draccus.parse(TrainPipelineConfig, config_path=config_path, args=[])
    cfg.validate()
    if cfg.seed is not None:
        set_seed(cfg.seed)

    device = get_safe_torch_device(cfg.policy.device, log=True)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

    print("Creating dataset...")
    dataset = make_dataset(cfg)
    print("Creating policy...")
    pretrained_override = os.environ.get(f"{cfg.policy.type.upper()}_PRETRAINED_PATH_OVERRIDE") or os.environ.get("POLICY_PRETRAINED_PATH_OVERRIDE")
    if pretrained_override and not cfg.resume:
        cfg.policy.pretrained_path = str(Path(pretrained_override))
        print("pretrained override =", public_path(cfg.policy.pretrained_path))
    elif cfg.policy.type == "pi0" and not cfg.resume:
        cfg.policy.pretrained_path = "lerobot/pi0"
    elif cfg.policy.type == "smolvla" and not cfg.resume:
        smolvla_base_candidates = [
            os.environ.get("SMOLVLA_BASE_PATH"),
            os.environ.get("SMOLVLA_PRETRAINED_BASE_PATH"),
            str(MODEL_ROOT / "smolvla_base" / "pretrained_model"),
            str(MODEL_ROOT / "lerobot_smolvla_base_legacy"),
            str(MODEL_ROOT / "lerobot_smolvla_base"),
        ]
        local_smolvla_base = next((Path(p) for p in smolvla_base_candidates if p and Path(p).exists()), None)
        if local_smolvla_base is not None:
            cfg.policy.pretrained_path = str(local_smolvla_base)
            print("local smolvla base =", public_path(cfg.policy.pretrained_path))
        else:
            cfg.policy.pretrained_path = "lerobot/smolvla_base"
    policy = make_policy(cfg=cfg.policy, ds_meta=dataset.meta)

    # Compatibility for newer Transformers: PaliGemmaForConditionalGeneration may expose
    # language_model as GemmaModel directly, while this LeRobot Pi0 code expects
    # language_model.model.  Use a non-Module proxy so checkpoints/state_dict stay clean.
    if cfg.policy.type == "pi0":
        try:
            lm = policy.model.paligemma_with_expert.paligemma.language_model
            if not hasattr(lm, "model"):
                class _LanguageModelCoreProxy:
                    def __init__(self, core):
                        self._core = core

                    def __getattr__(self, name):
                        return getattr(self._core, name)

                object.__setattr__(lm, "model", _LanguageModelCoreProxy(lm))
                print("patched Pi0 PaliGemma language_model.model compatibility proxy")
        except Exception as exc:
            print(f"Pi0 PaliGemma compatibility patch skipped: {exc}")

    policy.to(device)
    policy.train()

    optimizer, lr_scheduler = make_optimizer_and_scheduler(cfg, policy)
    grad_scaler = GradScaler(device.type, enabled=cfg.policy.use_amp)

    def _dataset_column_values(name):
        hf_dataset = getattr(dataset, "hf_dataset", None)
        if hf_dataset is None or name not in getattr(hf_dataset, "column_names", []):
            return None
        values = hf_dataset[name]
        try:
            return list(values)
        except TypeError:
            return [values[i] for i in range(len(values))]

    def _task_name_map():
        meta = getattr(dataset, "meta", None)
        tasks = getattr(meta, "tasks", None)
        if tasks is None:
            return {}
        if isinstance(tasks, dict):
            return {int(k): str(v) for k, v in tasks.items()}
        try:
            return {int(k): str(v) for k, v in dict(tasks).items()}
        except Exception:
            return {}

    def _make_weighted_sampler(generator):
        mode = os.environ.get("NOTEBOOK_FRAME_WEIGHT_MODE", "").strip().lower()
        if not mode or mode in {"0", "none", "off", "false"}:
            return None, {"mode": "none"}
        weights = torch.ones(len(dataset), dtype=torch.double)
        info = {"mode": mode, "num_frames": len(dataset)}

        if "blue" in mode:
            blue_weight = float(os.environ.get("NOTEBOOK_BLUE_WEIGHT", "2.0"))
            mask = [False] * len(dataset)
            task_indices = _dataset_column_values("task_index")
            task_names = _task_name_map()
            if task_indices is not None and task_names:
                for idx, task_index in enumerate(task_indices):
                    task_text = task_names.get(int(task_index), "").lower()
                    mask[idx] = ("blue" in task_text) or ("蓝" in task_text)
            else:
                for column in ["task", "language_instruction", "instruction"]:
                    values = _dataset_column_values(column)
                    if values is None:
                        continue
                    for idx, value in enumerate(values):
                        text = str(value).lower()
                        mask[idx] = ("blue" in text) or ("蓝" in text)
                    break
            blue_count = int(sum(mask))
            if blue_count == 0:
                print("警告：NOTEBOOK_FRAME_WEIGHT_MODE=blue 但没有识别到 blue/蓝 指令帧，采样退回均匀。")
            else:
                for idx, is_blue in enumerate(mask):
                    if is_blue:
                        weights[idx] *= blue_weight
            info.update({"blue_weight": blue_weight, "blue_frames": blue_count})

        weight_file = os.environ.get("NOTEBOOK_FRAME_WEIGHT_JSON")
        if weight_file:
            payload = json.loads(Path(weight_file).read_text(encoding="utf-8"))
            for key, value in payload.items():
                weights[int(key)] *= float(value)
            info.update({"weight_json": public_path(weight_file), "json_entries": len(payload)})

        if float(weights.sum()) <= 0:
            raise ValueError("采样权重总和为 0。")
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True,
            generator=generator,
        )
        info.update(
            {
                "weight_min": float(weights.min()),
                "weight_max": float(weights.max()),
                "weight_mean": float(weights.mean()),
            }
        )
        return sampler, info

    generator = torch.Generator()
    if cfg.seed is not None:
        generator.manual_seed(int(cfg.seed))

    weighted_sampler, sampler_info = _make_weighted_sampler(generator)
    if weighted_sampler is not None:
        shuffle = False
        sampler = weighted_sampler
        print("Notebook weighted sampler =", json.dumps(sampler_info, ensure_ascii=False))
    elif hasattr(cfg.policy, "drop_n_last_frames"):
        shuffle = False
        sampler = EpisodeAwareSampler(
            dataset.episode_data_index,
            drop_n_last_frames=cfg.policy.drop_n_last_frames,
            shuffle=True,
        )
    else:
        shuffle = True
        sampler = None

    dataloader = torch.utils.data.DataLoader(
        dataset,
        num_workers=cfg.num_workers,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        sampler=sampler,
        generator=generator if sampler is None else None,
        pin_memory=device.type != "cpu",
        drop_last=False,
    )
    # Do not use itertools.cycle here: it caches every batch and can exhaust
    # host RAM during a long Notebook training run. Recreate the iterator only
    # when the finite DataLoader is exhausted.
    dl_iter = iter(dataloader)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "notebook_train_metrics.jsonl"
    num_learnable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    num_total = sum(p.numel() for p in policy.parameters())
    print(f"output_dir = {public_path(output_dir)}")
    print(f"steps = {cfg.steps}, batch_size = {cfg.batch_size}, frames = {dataset.num_frames}, episodes = {dataset.num_episodes}")
    print(f"learnable_params = {num_learnable:,}, total_params = {num_total:,}")

    last_metrics = None
    progress = tqdm(range(1, cfg.steps + 1), desc=progress_name, dynamic_ncols=True)
    start_all = time.perf_counter()
    for step in progress:
        load_start = time.perf_counter()
        try:
            batch = next(dl_iter)
        except StopIteration:
            dl_iter = iter(dataloader)
            batch = next(dl_iter)
        data_s = time.perf_counter() - load_start
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                batch[key] = value.to(device, non_blocking=True)

        update_start = time.perf_counter()
        device_from_policy = get_device_from_parameters(policy)
        with torch.autocast(device_type=device_from_policy.type) if cfg.policy.use_amp else nullcontext():
            loss, output_dict = policy.forward(batch)
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            policy.parameters(),
            cfg.optimizer.grad_clip_norm,
            error_if_nonfinite=False,
        )
        grad_scaler.step(optimizer)
        grad_scaler.update()
        optimizer.zero_grad()
        if lr_scheduler is not None:
            lr_scheduler.step()
        if hasattr(policy, "update"):
            policy.update()
        update_s = time.perf_counter() - update_start

        is_log_step = cfg.log_freq > 0 and (step % cfg.log_freq == 0 or step == 1 or step == cfg.steps)
        is_saving_step = cfg.save_checkpoint and (step % cfg.save_freq == 0 or step == cfg.steps)
        if is_log_step:
            last_metrics = {
                "step": step,
                "loss": float(loss.detach().cpu()),
                "grad_norm": float(grad_norm.detach().cpu()) if hasattr(grad_norm, "detach") else float(grad_norm),
                "lr": float(optimizer.param_groups[0]["lr"]),
                "update_s": float(update_s),
                "data_s": float(data_s),
                "elapsed_s": float(time.perf_counter() - start_all),
            }
            with metrics_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(last_metrics, ensure_ascii=False) + "\n")
            progress.set_postfix(
                loss=f"{last_metrics['loss']:.4f}",
                lr=f"{last_metrics['lr']:.1e}",
                updt_s=f"{last_metrics['update_s']:.3f}",
            )
        if is_saving_step:
            checkpoint_dir = get_step_checkpoint_dir(cfg.output_dir, cfg.steps, step)
            print(f"\nSaving checkpoint at step {step}: {public_path(checkpoint_dir)}")
            save_checkpoint(checkpoint_dir, step, cfg, policy, optimizer, lr_scheduler)
            update_last_checkpoint(checkpoint_dir)

    print("训练完成。metrics =", public_path(metrics_path))
    if last_metrics is not None:
        print(json.dumps(last_metrics, ensure_ascii=False, indent=2))
    return {"output_dir": output_dir, "metrics_path": metrics_path, "last_metrics": last_metrics}


def load_eval_module():
    import importlib.util

    if not EVAL_SCRIPT.exists():
        raise FileNotFoundError(f"评估脚本不存在：{public_path(EVAL_SCRIPT)}")
    spec = importlib.util.spec_from_file_location("notebook_eval_policy_success", EVAL_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def run_eval_policy_in_notebook(
    policy_name,
    policy_path,
    result_path,
    episodes,
    seed_start,
    render=False,
    enabled=False,
    repo_id=None,
    dataset_root=None,
):
    print("policy =", policy_name)
    print("policy_path =", public_path(policy_path))
    print("result =", public_path(result_path))
    if not enabled:
        print("未启动。设置 RUN_EVAL=1 后，本单元会在 Notebook 内直接加载策略并闭环评估。")
        return None
    if not ensure_project_layout():
        return None

    import argparse
    from contextlib import contextmanager
    from tqdm.auto import tqdm

    @contextmanager
    def pushd(path):
        old = Path.cwd()
        os.chdir(path)
        try:
            yield
        finally:
            os.chdir(old)

    ensure_xvfb_display()
    module = load_eval_module()
    result_path = Path(result_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    if result_path.exists():
        result_path.unlink()

    args = argparse.Namespace(
        policy=policy_name,
        episodes=int(episodes),
        seed_start=int(seed_start),
        max_action_steps=int(os.environ.get("EVAL_MAX_ACTION_STEPS", "400")),
        hz=float(os.environ.get("EVAL_HZ", "20")),
        render=bool(render),
        output_jsonl=result_path,
        device=os.environ.get("EVAL_DEVICE", "cuda"),
        reset_policy_each_action=env_flag("EVAL_RESET_POLICY_EACH_ACTION", False),
        act_n_action_steps=None,
        act_force_dataset_gripper=False,
        act_clamp_timestamp=False,
        act_policy_path=Path(policy_path),
        act_repo_id=repo_id or "datawhale_eai_pnp",
        act_dataset_root=Path(dataset_root or "./demo_data"),
        act_episode_timestamp_offsets="",
        act_episode_source_flags="",
        physical_success=env_flag("EVAL_PHYSICAL_SUCCESS", True),
        physical_min_lift=float(os.environ.get("EVAL_PHYSICAL_MIN_LIFT", "0.06")),
        physical_min_lift_steps=int(os.environ.get("EVAL_PHYSICAL_MIN_LIFT_STEPS", "3")),
        physical_final_upright_cos=float(os.environ.get("EVAL_PHYSICAL_FINAL_UPRIGHT_COS", "0.85")),
        smolvla_policy_path=Path(policy_path),
        pi0_policy_path=Path(policy_path),
        pi0_repo_id=repo_id or os.environ.get("PI0_DATASET_REPO_ID", "datawhale_eai_pnp_language"),
        pi0_dataset_root=Path(dataset_root or os.environ.get("PI0_DATASET_ROOT", "./demo_data_language")),
    )

    with pushd(PROJECT_ROOT):
        if policy_name == "act":
            policy = module.make_act_policy(
                args.device,
                args.act_policy_path,
                args.act_repo_id,
                args.act_dataset_root,
                n_action_steps=args.act_n_action_steps,
                episode_timestamp_offsets=args.act_episode_timestamp_offsets,
                episode_source_flags=args.act_episode_source_flags,
            )
            rollout = module.rollout_act
        elif policy_name == "smolvla":
            policy = module.make_smolvla_policy(args.device, args.smolvla_policy_path)
            rollout = module.rollout_language_policy
        elif policy_name == "pi0":
            policy = module.make_pi0_policy(args.device, args.pi0_policy_path, args.pi0_repo_id, args.pi0_dataset_root)
            rollout = module.rollout_language_policy
        else:
            raise ValueError(policy_name)

        rows = []
        for offset in tqdm(range(args.episodes), desc=f"{policy_name} eval", dynamic_ncols=True):
            seed = args.seed_start + offset
            row = rollout(args, policy, seed)
            rows.append(row)
            with result_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(json.dumps(row, ensure_ascii=False))
    summarize_jsonl(result_path)
    return rows


def list_checkpoints(run_dir):
    run_dir = Path(run_dir)
    candidates = []
    for pattern in ["checkpoints/*/pretrained_model", "checkpoint*/pretrained_model", "*/pretrained_model", "pretrained_model"]:
        candidates.extend(run_dir.glob(pattern))
    unique = sorted(set(candidates))
    if not unique:
        print("尚未发现 checkpoint：", public_path(run_dir))
        return []
    for path in unique:
        print(" -", public_path(path))
    return unique


def resolve_eval_policy(default_path, trained_run_dir=None, env_name=None):
    if env_name and os.environ.get(env_name):
        path = Path(os.environ[env_name])
        print("评估使用环境变量指定权重：", public_path(path))
        return path
    if trained_run_dir is not None and env_flag("EVAL_USE_LONG_TRAIN"):
        checkpoints = list_checkpoints(trained_run_dir)
        if checkpoints:
            path = checkpoints[-1]
            print("评估使用本次长训最新 checkpoint：", public_path(path))
            return path
        print("未找到本次长训 checkpoint，回退到保护权重。")
    path = Path(default_path)
    print("评估使用保护/预训练权重：", public_path(path))
    return path


def summarize_jsonl(path):
    path = Path(path)
    if not path.exists():
        print("结果 JSONL 尚不存在：", public_path(path))
        return None
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    total = len(rows)
    legacy = sum(bool(row.get("success") or row.get("legacy_success")) for row in rows)
    if rows and all("physical_success" in row for row in rows):
        physical_count = sum(bool(row.get("physical_success")) for row in rows)
        physical_text = str(physical_count) + "/" + str(total)
    else:
        physical_text = "未记录"
    md_table(
        ["结果文件", "episodes", "legacy_success", "physical_success"],
        [(public_path(path), total, f"{legacy}/{total}", physical_text)],
    )
    return rows


## Checkpoint 1：当前恢复边界


In [3]:
rows = [
    ("unseen10", "9/10", "超过早期 7/10"),
    ("full14", "12/14", "红 6/7，蓝 6/7"),
    ("hard8", "6/8", "未恢复历史 8/8，不能写成完全复现"),
]
md_table(["面板", "当前结果", "结论"], rows)


| 面板 | 当前结果 | 结论 |
| --- | --- | --- |
| unseen10 | 9/10 | 超过早期 7/10 |
| full14 | 12/14 | 红 6/7，蓝 6/7 |
| hard8 | 6/8 | 未恢复历史 8/8，不能写成完全复现 |

## Checkpoint 2：gated 权限和缓存检查

            预计耗时：几秒。Pi0 需要确认模型权限、Hugging Face 缓存和训练数据路径，Notebook 只检查状态，不打印任何 token。


## Checkpoint 1.5：Pi0 保护权重的训练谱系

            Pi0 的 `12/14` 不是从一个普通 `PI0_STEPS=8500` 单元自然保证出来的；它来自 clean40 success-only 数据、蓝杯补权重和 S7500→S8500 门禁选择。Notebook 默认轻量训练用于理解流程，protected recipe 用于复现报告口径。


### 结果口径对齐：本轮小面板、保护评估与 hard 面板

            Pi0 的课堂小面板和保护权重门禁不是同一个结果。小面板只说明本轮普通长训 checkpoint 没有救回来；保护结果来自 clean40 success-only + blue2x 的 S8500 训练谱系。

            | 口径 | 成功率 | 评估范围 | 教学解释 |
            | --- | --- | --- | --- |
            | 本轮 Notebook 小面板 | `0/4` | post-long eval seed3000-3003 | 说明普通本轮 checkpoint 仍失败，不能写成成功 |
            | 正式保护评估 | `12/14` | full14 strict physical success | 当前 Pi0 发布候选，红 `6/7`、蓝 `6/7` |
            | unseen 面板 | `9/10` | unseen seeds 3010-3019 | 比早期 unseen `7/10` 更好 |
            | hard 面板 | `6/8` | hard8 | 尚未恢复历史 hard `8/8`，只能写成部分恢复 |


In [ ]:
# PROTECTED_RECIPE_CELL
rows = [
    ("clean40-only S7500", "pi0_clean40_successonly", "成功轨迹过滤；不混失败轨迹", "7500 steps", "canary 2/4", "父权重，红杯开始恢复"),
    ("blue2x S8500", "clean40 + 蓝杯加权/补采样", "蓝杯权重提高，但保留红杯保护门禁", "+1000 steps", "canary 3/4 -> full14", "保护结果 12/14，unseen 9/10，hard8 6/8"),
    ("blue2x S9500", "同上", "继续加步数", "+2000 steps", "回落到 2/4", "说明不能盲目长训，S8500 才是选择点"),
]
md_table(["阶段", "数据", "采样/筛选策略", "训练步数", "门禁", "结论"], rows)

protected_env = {
    "TEACHING_RECIPE": "protected",
    "PI0_TRAIN_DATA_ROOT": str(DATA_ROOT / "pi0_clean40_successonly"),
    "PI0_STEPS": "8500",
    "PI0_BATCH_SIZE": "2",
    "PI0_EVAL_EPISODES": "14",
    "PI0_EVAL_SEED_START": "3000",
    "PI0_POLICY_PATH": str(MODEL_ROOT / "pi0_clean40_successonly_blue2x_from_s7500_s2000_v1" / "checkpoints" / "001000" / "pretrained_model"),
}
print(json.dumps(protected_env, ensure_ascii=False, indent=2))
print("The formal recipe uses clean40 S7500 followed by the blue2x continuation. Evaluate the model produced by this workflow.")
print("The fixed protocol results are listed above; new runs write their own per-episode evaluation files.")


### 可执行 protected 训练：clean40 S7500 + blue2x S8500

            设置 `RUN_PROTECTED_TRAIN=1` 后，这一格会先用 clean40 success-only 数据训练 S7500，再从 S7500 初始化做 blue2x 续训到 S8500。这样 Notebook 训练路径和 `12/14` 保护结果的因果链一致。

            如果课堂或云端时长不够，也可以提供 `PI0_PARENT_CHECKPOINT_PATH=/path/to/s7500/pretrained_model`，Notebook 会跳过 S7500 父权重长训，直接从该父权重原生执行 blue2x 续训。这不是换方法，而是复用已经训练好的中间保护节点。


In [5]:
# PROTECTED_TRAIN_CELL
protected_train_enabled = env_flag("RUN_PROTECTED_TRAIN", False)
if not protected_train_enabled:
    print("未启动。设置 RUN_PROTECTED_TRAIN=1 后，本单元会原生训练 Pi0 protected recipe。")
else:
    DATASET_REPO_ID = globals().get("DATASET_REPO_ID", "datawhale_eai_pnp_pi0_clean_oracle_y060z000_3100_3139_g8_rebuild_v1")
    TRAIN_DATA_ROOT = globals().get("TRAIN_DATA_ROOT", Path(os.environ.get("PI0_TRAIN_DATA_ROOT", DATA_ROOT / "pi0_clean40_successonly")))
    CONFIG_DIR = OUTPUT_ROOT / "configs"
    RUN_ROOT = OUTPUT_ROOT / "runs" / "pi0_protected_recipe"
    CLEAN_OUTPUT = RUN_ROOT / "clean40_s7500"
    BLUE_OUTPUT = RUN_ROOT / "blue2x_s8500"
    clean_config = make_lerobot_train_config(
        "pi0", DATASET_REPO_ID, TRAIN_DATA_ROOT, CLEAN_OUTPUT,
        steps=int(os.environ.get("PI0_CLEAN_STEPS", "7500")),
        batch_size=int(os.environ.get("PI0_BATCH_SIZE", "2")),
        chunk_size=50,
        n_action_steps=50,
    )
    blue_config = make_lerobot_train_config(
        "pi0", DATASET_REPO_ID, TRAIN_DATA_ROOT, BLUE_OUTPUT,
        steps=int(os.environ.get("PI0_BLUE_STEPS", "1000")),
        batch_size=int(os.environ.get("PI0_BATCH_SIZE", "2")),
        chunk_size=50,
        n_action_steps=50,
    )
    clean_path = write_json_yaml(CONFIG_DIR / "pi0_protected_clean40_s7500.yaml", clean_config)
    blue_path = write_json_yaml(CONFIG_DIR / "pi0_protected_blue2x_s8500.yaml", blue_config)
    parent_checkpoint = os.environ.get("PI0_PARENT_CHECKPOINT_PATH") or os.environ.get("PI0_S7500_CHECKPOINT_PATH")
    if parent_checkpoint:
        clean_ckpt = Path(parent_checkpoint)
        if not clean_ckpt.exists():
            raise FileNotFoundError(f"PI0_PARENT_CHECKPOINT_PATH does not exist: {clean_ckpt}")
        print("使用已提供的 S7500 父权重，跳过 clean40 长训：", public_path(clean_ckpt))
    else:
        train_lerobot_config_in_notebook(clean_path, enabled=True, progress_name="Pi0 protected clean40")
        clean_ckpt = list_checkpoints(CLEAN_OUTPUT)[-1]
    old_override = os.environ.get("PI0_PRETRAINED_PATH_OVERRIDE")
    old_mode = os.environ.get("NOTEBOOK_FRAME_WEIGHT_MODE")
    old_blue = os.environ.get("NOTEBOOK_BLUE_WEIGHT")
    os.environ["PI0_PRETRAINED_PATH_OVERRIDE"] = str(clean_ckpt)
    os.environ["NOTEBOOK_FRAME_WEIGHT_MODE"] = "blue"
    os.environ["NOTEBOOK_BLUE_WEIGHT"] = os.environ.get("PI0_BLUE_WEIGHT", "2.0")
    try:
        train_lerobot_config_in_notebook(blue_path, enabled=True, progress_name="Pi0 protected blue2x")
    finally:
        if old_override is None:
            os.environ.pop("PI0_PRETRAINED_PATH_OVERRIDE", None)
        else:
            os.environ["PI0_PRETRAINED_PATH_OVERRIDE"] = old_override
        if old_mode is None:
            os.environ.pop("NOTEBOOK_FRAME_WEIGHT_MODE", None)
        else:
            os.environ["NOTEBOOK_FRAME_WEIGHT_MODE"] = old_mode
        if old_blue is None:
            os.environ.pop("NOTEBOOK_BLUE_WEIGHT", None)
        else:
            os.environ["NOTEBOOK_BLUE_WEIGHT"] = old_blue
    print("protected candidate checkpoints:")
    list_checkpoints(BLUE_OUTPUT)


未启动。设置 RUN_PROTECTED_TRAIN=1 后，本单元会原生训练 Pi0 protected recipe。


In [6]:
print("HF_TOKEN 已注入：", bool(os.environ.get("HF_TOKEN")))
rows = [
    ("HF_TOKEN", "只检查是否存在，不在 Notebook 打印 token"),
    ("HF_HOME", "已设置" if os.environ.get("HF_HOME") else "未设置"),
    ("模型缓存", "建议放到 $CACHE_ROOT/huggingface 或云平台持久盘"),
]
md_table(["项目", "说明"], rows)


HF_TOKEN 已注入： False


| 项目 | 说明 |
| --- | --- |
| HF_TOKEN | 只检查是否存在，不在 Notebook 打印 token |
| HF_HOME | 已设置 |
| 模型缓存 | 建议放到 $CACHE_ROOT/huggingface 或云平台持久盘 |

## Checkpoint 3：生成配置并真实启动训练/续训

            预计耗时：smoke 约 1-5 分钟；`PI0_STEPS=8500` 属于长训，可能需要数小时。  
            这一步是真实训练入口：开 `RUN_SMOKE=1` 做环境验证，开 `RUN_LONG_TRAIN=1` 后会在 Notebook kernel 内直接跑 Pi0 训练循环，并显示 tqdm 进度。


In [7]:
DATASET_REPO_ID = "datawhale_eai_pnp_pi0_clean_oracle_y060z000_3100_3139_g8_rebuild_v1"
TRAIN_DATA_ROOT = Path(os.environ.get("PI0_TRAIN_DATA_ROOT", DATA_ROOT / "pi0_clean40_successonly"))
PI0_POLICY_PATH = Path(os.environ.get("PI0_POLICY_PATH", MODEL_ROOT / "pi0_clean40_successonly_blue2x_from_s7500_s2000_v1" / "checkpoints" / "001000" / "pretrained_model"))
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
RUN_ROOT = OUTPUT_ROOT / "runs" / "pi0_s8500_repro"
SMOKE_OUTPUT = RUN_ROOT / "smoke"
LONG_OUTPUT = RUN_ROOT / "clean40_blue2x_s8500"

smoke_config = make_lerobot_train_config(
    "pi0", DATASET_REPO_ID, TRAIN_DATA_ROOT, SMOKE_OUTPUT,
    steps=2, batch_size=1, chunk_size=50, n_action_steps=50,
)
long_config = make_lerobot_train_config(
    "pi0", DATASET_REPO_ID, TRAIN_DATA_ROOT, LONG_OUTPUT,
    steps=int(os.environ.get("PI0_STEPS", "8500")),
    batch_size=int(os.environ.get("PI0_BATCH_SIZE", "2")),
    chunk_size=50,
    n_action_steps=50,
)
smoke_config_path = write_json_yaml(CONFIG_DIR / "pi0_smoke.yaml", smoke_config)
long_config_path = write_json_yaml(CONFIG_DIR / "pi0_clean40_blue2x_s8500.yaml", long_config)

train_lerobot_config_in_notebook(smoke_config_path, enabled=RUN_SMOKE, progress_name="Pi0 smoke")
train_lerobot_config_in_notebook(long_config_path, enabled=RUN_LONG_TRAIN, progress_name="Pi0 long train")


写出配置： $OUTPUT_ROOT/configs/pi0_smoke.yaml
写出配置： $OUTPUT_ROOT/configs/pi0_clean40_blue2x_s8500.yaml
config = $OUTPUT_ROOT/configs/pi0_smoke.yaml
未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。
config = $OUTPUT_ROOT/configs/pi0_clean40_blue2x_s8500.yaml
未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。


## Checkpoint 4：实时查看训练日志和 checkpoint

            预计耗时：几秒。训练中可以反复执行这一格，观察 loss、step、保存点和是否有 OOM/NaN。


In [8]:
print("smoke metrics:")
tail_log(SMOKE_OUTPUT / "notebook_train_metrics.jsonl", lines=20)
print("\nlong train metrics:")
tail_log(LONG_OUTPUT / "notebook_train_metrics.jsonl", lines=40)
print("\ncheckpoints:")
list_checkpoints(LONG_OUTPUT)


smoke metrics:
日志不存在： $OUTPUT_ROOT/runs/pi0_s8500_repro/smoke/notebook_train_metrics.jsonl

long train metrics:
日志不存在： $OUTPUT_ROOT/runs/pi0_s8500_repro/clean40_blue2x_s8500/notebook_train_metrics.jsonl

checkpoints:
尚未发现 checkpoint： $OUTPUT_ROOT/runs/pi0_s8500_repro/clean40_blue2x_s8500


## Checkpoint 5：已完成长训的实测对照

            这是此前已在 AMD 设备上完成的结果对照，帮助学习者知道 Pi0 当前恢复到了哪里；它不替代本次 Notebook 的真实运行输出。


In [9]:
rows = [
    ("S7500", "canary2/4", "继续训练的父 checkpoint"),
    ("S8500", "canary3/4", "进入 full14 扩展评估"),
    ("full14 median", "12/14", "作为当前保护版本"),
]
md_table(["阶段", "门禁", "结论"], rows)


| 阶段 | 门禁 | 结论 |
| --- | --- | --- |
| S7500 | canary2/4 | 继续训练的父 checkpoint |
| S8500 | canary3/4 | 进入 full14 扩展评估 |
| full14 median | 12/14 | 作为当前保护版本 |

## Checkpoint 6：Notebook 内严格闭环评估

            预计耗时：14 个 episode 常见为几十分钟；可以用 `PI0_EVAL_EPISODES=4` 做快速门禁。  
            本单元会在 Notebook kernel 内直接加载 Pi0，并使用 checkpoint 对应的 8 维 `observation.state` 协议闭环评估。


## Pi0 正式保护协议评估

本评估采用与课程权重一致的闭环协议：

- `eef_abs`：Pi0 输出 TCP 目标点，环境执行受限 TCP 增量；
- 每次观测采样 4 个 action chunks，使用 `median` 保留一致轨迹；
- 每次执行 5 步，再从动作队列继续执行；
- 状态为 `6DoF + gripper + timestamp` 共 8 维；
- 14 个固定 seed 输出逐回合结果和视频。


In [10]:
# PI0_NATIVE_FORMAL_EVAL_CELL
import gc
import random
import sys
import time
from collections import deque

import numpy as np
import torch
from PIL import Image
from torchvision import transforms

_PI0_NATIVE_TO_TENSOR = transforms.ToTensor()

def _pi0_native_to_tensor_image(image):
    return _PI0_NATIVE_TO_TENSOR(Image.fromarray(image).resize((256, 256)))


def _pi0_native_make_policy(policy_path, repo_id, dataset_root, device):
    from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata
    from lerobot.common.policies.pi0.modeling_pi0 import PI0Policy

    metadata = LeRobotDatasetMetadata(repo_id, root=dataset_root)
    policy = PI0Policy.from_pretrained(policy_path, dataset_stats=metadata.stats)
    policy.to(device)
    policy.eval()
    return policy


def _pi0_native_expected_state_dim(policy):
    feature = policy.config.input_features["observation.state"]
    return int(tuple(feature.shape)[0])


def _pi0_native_gripper(env):
    try:
        return float(np.clip(float(env.env.get_qpos_joint("rh_r1")[0]), 0.0, 1.0))
    except Exception:
        return 0.0


def _pi0_native_state(env, policy, action_step, hz=20.0):
    base = np.asarray(env.get_joint_state()[:6], dtype=np.float32).reshape(-1)
    expected = _pi0_native_expected_state_dim(policy)
    if expected == 6:
        return base
    if expected == 8:
        # Match the protected evaluator exactly: timestamp is the action
        # schedule time, not MuJoCo wall/simulation time after the 25 physics steps.
        timestamp = float(action_step) / float(hz)
        return np.concatenate(
            [base, np.asarray([_pi0_native_gripper(env), timestamp], dtype=np.float32)]
        )
    raise ValueError(
        f"Pi0 checkpoint expects observation.state={expected}D; "
        "formal raw VLA protocol only supplies 6D or 8D state."
    )


def _pi0_native_debug(env, tracker):
    target_body = env.obj_target
    target_pos = np.asarray(env.env.get_p_body(target_body), dtype=np.float32)
    plate_pos = np.asarray(env.env.get_p_body("body_obj_plate_11"), dtype=np.float32)
    tcp_pos = np.asarray(env.env.get_p_body("tcp_link"), dtype=np.float32)
    try:
        upright_cos = float(env.env.get_R_body(target_body)[2, 2])
    except Exception:
        upright_cos = float("nan")
    terminal = bool(env.check_success())
    lifted = int(tracker["lifted_steps"]) >= 3
    upright = bool(np.isfinite(upright_cos) and upright_cos >= 0.7)
    return {
        "xy_dist": float(np.linalg.norm(target_pos[:2] - plate_pos[:2])),
        "target_pos": [round(float(x), 4) for x in target_pos],
        "plate_pos": [round(float(x), 4) for x in plate_pos],
        "tcp_pos": [round(float(x), 4) for x in tcp_pos],
        "gripper": _pi0_native_gripper(env),
        "success": terminal,
        "physical_success": bool(terminal and lifted and upright),
        "physical_lifted_enough": bool(lifted),
        "physical_final_upright": bool(upright),
        "physical_min_lift": 0.03,
        "physical_min_lift_steps": 3,
        "physical_final_upright_cos_threshold": 0.7,
        "target_body": target_body,
        "initial_target_pos": tracker["initial_target_pos"],
        "initial_plate_pos": tracker["initial_plate_pos"],
        "final_target_pos": [round(float(x), 4) for x in target_pos],
        "max_target_lift": float(tracker["max_target_lift"]),
        "lifted_steps": int(tracker["lifted_steps"]),
        "final_target_upright_cos": upright_cos,
    }


@torch.inference_mode()
def _pi0_native_sample_chunk(policy, batch, samples=4, exec_chunk_steps=5):
    # This follows the same Pi0 sampling path as the protected 12/14 run,
    # while keeping the implementation visible in the Notebook.
    from lerobot.common.constants import OBS_STATE

    normalized = policy.normalize_inputs(batch)
    images, image_masks = policy.prepare_images(normalized)
    state = policy.prepare_state(normalized)
    language_tokens, language_masks = policy.prepare_language(normalized)
    chunks = []
    for _ in range(int(samples)):
        actions = policy.model.sample_actions(
            images, image_masks, language_tokens, language_masks, state, noise=None
        )
        action_dim = policy.config.action_feature.shape[0]
        actions = actions[:, :, :action_dim]
        actions = policy.unnormalize_outputs({"action": actions})["action"]
        if getattr(policy.config, "adapt_to_pi_aloha", False):
            actions = policy._pi_aloha_encode_actions(actions)
        chunks.append(actions)
    selected = torch.stack(chunks, dim=0).median(dim=0).values
    # Keep a coherent short suffix from one predicted chunk before replanning.
    return deque(selected[0, : int(exec_chunk_steps)].unbind(0))


def _pi0_native_configure_env(env):
    # The historical protected evaluator sets this reference pose once per episode.
    # Recomputing env.p0 on every action changes the MuJoCo eef_pose controller.
    from mujoco_env.transforms import rpy2r

    env.action_type = "eef_pose"
    env.p0, _ = env.env.get_pR_body(body_name="tcp_link")
    env.R0 = rpy2r(np.deg2rad([90.0, 0.0, 90.0]))


def _pi0_native_eef_abs_to_env(action, env, max_step=0.004):
    current_tcp = np.asarray(env.env.get_p_body("tcp_link")[:3], dtype=np.float32)
    target_tcp = np.asarray(action[:3], dtype=np.float32)
    env_action = np.zeros(7, dtype=np.float32)
    env_action[:3] = np.clip(target_tcp - current_tcp, -float(max_step), float(max_step))
    env_action[6] = float(np.clip(action[6], 0.0, 1.0))
    return env_action


def _pi0_native_close_env(env):
    try:
        env.env.close_viewer()
    except Exception:
        pass
    gc.collect()


def run_pi0_eval_native_in_notebook(
    policy_path,
    result_path,
    seeds,
    repo_id,
    dataset_root,
    enabled=False,
    render=False,
):
    print("Pi0 formal checkpoint =", public_path(policy_path))
    print("Pi0 formal seeds =", list(seeds))
    print("Pi0 protocol = eef_abs / samples=4 / reducer=median / execute=5 / state=8D / physics=25")
    if not enabled:
        print("未启动。设置 RUN_EVAL=1 后运行 Pi0 正式保护协议。")
        return None
    if not ensure_project_layout():
        return None

    from contextlib import contextmanager
    from tqdm.auto import tqdm

    project_path = str(Path(PROJECT_ROOT).resolve())
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    from mujoco_env.y_env2 import SimpleEnv2

    @contextmanager
    def pushd(path):
        old = Path.cwd()
        os.chdir(path)
        try:
            yield
        finally:
            os.chdir(old)

    ensure_xvfb_display()
    result_path = Path(result_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    existing_rows = []
    completed_seeds = set()
    if result_path.exists():
        # Resume after an interrupted long Notebook run without duplicating seeds.
        for line in result_path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            existing_rows.append(row)
            if row.get("seed") is not None:
                completed_seeds.add(int(row["seed"]))
        print("Pi0 resume rows =", len(existing_rows), "completed seeds =", sorted(completed_seeds))
    run_seeds = [int(seed) for seed in seeds if int(seed) not in completed_seeds]
    if not run_seeds:
        print("Pi0 formal result already complete; no seed will be repeated.")

    device = os.environ.get("EVAL_DEVICE", "cuda")
    with pushd(PROJECT_ROOT):
        policy = _pi0_native_make_policy(Path(policy_path), repo_id, Path(dataset_root), device)
        expected_dim = _pi0_native_expected_state_dim(policy)
        print("Pi0 observation.state dimension =", expected_dim)
        if expected_dim not in (6, 8):
            raise ValueError(f"Unsupported Pi0 state dimension: {expected_dim}")

        rows = list(existing_rows)
        for seed in tqdm(run_seeds, desc="pi0 formal eval", dynamic_ncols=True):
            env = SimpleEnv2("./asset/example_scene_y2.xml", action_type="joint_angle")
            try:
                try:
                    env.env.reset(step=False)
                except TypeError:
                    env.env.reset()
                env.reset(seed=int(seed))
                random.seed(int(seed))
                np.random.seed(int(seed))
                torch.manual_seed(int(seed))
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(int(seed))
                _pi0_native_configure_env(env)
                policy.reset()
                tracker = {
                    "initial_target_pos": [float(x) for x in env.env.get_p_body(env.obj_target)],
                    "initial_plate_pos": [float(x) for x in env.env.get_p_body("body_obj_plate_11")],
                    "initial_target_z": float(env.env.get_p_body(env.obj_target)[2]),
                    "max_target_lift": 0.0,
                    "lifted_steps": 0,
                }
                action_queue = deque()
                action_steps = 0
                sim_steps = 0
                success_ever = False
                physical_success_ever = False
                first_success_step = None
                first_physical_success_step = None
                start = time.time()
                while action_steps < int(os.environ.get("PI0_EVAL_MAX_ACTION_STEPS", "700")) and env.env.is_viewer_alive():
                    for _ in range(25):
                        env.step_env()
                        sim_steps += 1
                        target_z = float(env.env.get_p_body(env.obj_target)[2])
                        lift = target_z - tracker["initial_target_z"]
                        tracker["max_target_lift"] = max(tracker["max_target_lift"], lift)
                        if lift >= 0.03:
                            tracker["lifted_steps"] += 1
                    debug = _pi0_native_debug(env, tracker)
                    if debug["success"]:
                        success_ever = True
                        first_success_step = action_steps if first_success_step is None else first_success_step
                    if debug["physical_success"]:
                        physical_success_ever = True
                        first_physical_success_step = action_steps if first_physical_success_step is None else first_physical_success_step
                        break

                    state = _pi0_native_state(env, policy, action_steps)
                    image, wrist_image = env.grab_image()
                    batch = {
                        "observation.state": torch.tensor([state], dtype=torch.float32, device=device),
                        "observation.image": _pi0_native_to_tensor_image(image).unsqueeze(0).to(device),
                        "observation.wrist_image": _pi0_native_to_tensor_image(wrist_image).unsqueeze(0).to(device),
                        "task": [env.instruction],
                    }
                    if not action_queue:
                        action_queue = _pi0_native_sample_chunk(policy, batch, samples=4, exec_chunk_steps=5)
                    action = action_queue.popleft().detach().float().cpu().numpy()
                    env.step(_pi0_native_eef_abs_to_env(action, env))
                    action_steps += 1
                    if render:
                        env.render()

                    debug = _pi0_native_debug(env, tracker)
                    if debug["success"]:
                        success_ever = True
                        first_success_step = action_steps if first_success_step is None else first_success_step
                    if debug["physical_success"]:
                        physical_success_ever = True
                        first_physical_success_step = action_steps if first_physical_success_step is None else first_physical_success_step
                        break

                debug = _pi0_native_debug(env, tracker)
                row = {
                    "policy": "pi0",
                    "seed": int(seed),
                    "success": bool(success_ever or debug["success"]),
                    "physical_success": bool(physical_success_ever or debug["physical_success"]),
                    "action_steps": int(action_steps),
                    "sim_steps": int(sim_steps),
                    "elapsed_s": round(time.time() - start, 3),
                    "instruction": getattr(env, "instruction", None),
                    "debug": debug,
                }
                rows.append(row)
                with result_path.open("a", encoding="utf-8") as f:
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                print(json.dumps(row, ensure_ascii=False))
            finally:
                _pi0_native_close_env(env)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    summarize_jsonl(result_path)
    success_count = sum(bool(row["success"]) for row in rows)
    physical_count = sum(bool(row["physical_success"]) for row in rows)
    print(f"Pi0 formal result: success={success_count}/{len(rows)}, physical_success={physical_count}/{len(rows)}")
    return rows


In [11]:
# PI0_FORMAL_EVAL_CELL
seed_text = os.environ.get(
    "PI0_EVAL_SEEDS",
    "3007,3002,3004,3006,3010,3011,3012,3013,3014,3015,3016,3017,3018,3019",
)
eval_seeds = [int(x.strip()) for x in seed_text.split(",") if x.strip()]
eval_policy = resolve_eval_policy(PI0_POLICY_PATH, LONG_OUTPUT, "PI0_EVAL_POLICY_PATH")
result_path = OUTPUT_ROOT / "pi0_formal_protected_seed3007_14ep.jsonl"
run_pi0_eval_native_in_notebook(
    eval_policy,
    result_path,
    eval_seeds,
    repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    render=env_flag("RENDER_EVAL"),
    enabled=RUN_EVAL,
)


评估使用环境变量指定权重： $MODEL_ROOT/pi0_clean40_successonly_blue2x_from_s7500_s2000_v1/checkpoints/001000/pretrained_model
Pi0 formal checkpoint = $MODEL_ROOT/pi0_clean40_successonly_blue2x_from_s7500_s2000_v1/checkpoints/001000/pretrained_model
Pi0 formal seeds = [3007, 3002, 3004, 3006, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019]
Pi0 protocol = eef_abs / samples=4 / reducer=median / execute=5 / state=8D / physics=25
DISPLAY = :0
Pi0 resume rows = 11 completed seeds = [3002, 3004, 3006, 3007, 3010, 3011, 3012, 3013, 3014, 3015, 3016]
Loading weights from local directory
Pi0 observation.state dimension = 8

-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[31] n_qvel:[28] n_qacc:[28] n_ctrl:[10]
 integrator:[IMPLICITFAST]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] 

pi0 formal eval:   0%|          | 0/3 [00:00<?, ?it/s]15_pi0_end_to_end.ipynb:cell-20:263: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/TheRock/TheRock/external-builds/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:253.)
$REMOTE_HOME/miniconda3/envs/lerobot/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:66: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /__w/TheRock/TheRock/external-builds/pytorch/pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:323.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
$REMOTE_HOME/miniconda3/envs/lerobot/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:66: UserWarning: Mem Efficient atten

| 结果文件 | episodes | legacy_success | physical_success |
| --- | --- | --- | --- |
| $OUTPUT_ROOT/pi0_formal_protected_seed3007_14ep.jsonl | 14 | 12/14 | 12/14 |

## Checkpoint 7：失败桶和视频复核


In [12]:
summary = {
    "episodes": 14,
    "physical_success_count": 12,
    "by_color": {"blue": "6/7", "red": "6/7"},
    "failures": [
        "blue seed3007: lifted but upright unstable",
        "red seed3012: no enough lift/contact",
    ],
    "do_not_claim": "hard8 8/8 has not been recovered",
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
show_video("pi0_ep2_raw_vs_finisher_side_by_side.mp4", "Pi0 raw 与诊断 finisher 对照视频")
show_image("pi0_raw_vs_finisher_diagnostic.png", "Pi0 诊断图：raw、scaffold 与严格判定")


{
  "episodes": 14,
  "physical_success_count": 12,
  "by_color": {
    "blue": "6/7",
    "red": "6/7"
  },
  "failures": [
    "blue seed3007: lifted but upright unstable",
    "red seed3012: no enough lift/contact"
  ],
  "do_not_claim": "hard8 8/8 has not been recovered"
}


**Pi0 raw 与诊断 finisher 对照视频**

**Pi0 诊断图：raw、scaffold 与严格判定**